<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/Topics/Parameter_Efficient_Finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [64]:
!pip install -U bitsandbytes>=0.46.1

In [65]:
!pip install trl

In [66]:
import torch
from torch import nn
import torch.nn.functional as F

import transformers
from tqdm import tqdm
device = 'cuda'

In [67]:
from transformers import BitsAndBytesConfig,AutoModelForCausalLM, TrainingArguments, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

model_name = 'unsloth/Qwen3-0.6B'


bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [68]:
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map = device)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float32
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [69]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024, padding_idx=151654)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (

In [70]:
peft_config = LoraConfig(
    r = 16,
    lora_alpha = 16*2,
    target_modules = ['q_proj' 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout = 0.05
)

In [71]:
!pip install --upgrade torchao

In [72]:
peft_model = get_peft_model(model, peft_config)

In [73]:
peft_model.print_trainable_parameters()

trainable params: 7,798,784 || all params: 603,848,704 || trainable%: 1.2915


In [74]:
from trl import SFTTrainer

In [75]:
instructions = ['Calculate 1+1', 'What is the capital of Uzbekistan']
answers = ['2', 'Tashkent']

In [76]:
messages_list = []
for x, y in zip(instructions, answers):
  messages_list.append({
    'messages':[
        {
            "role":"system",
            "content":"You are AI assistant which answers the question without a lot of details"
        },
        {
            "role":"user",
            "content":x
        },
        {
            "role":"assistant",
            "content": y
        }

    ]
})


In [77]:
messages_list

[{'messages': [{'role': 'user', 'content': 'Calculate 1+1'},
   {'role': 'assistant', 'content': '2'}]},
 {'messages': [{'role': 'user',
    'content': 'What is the capital of Uzbekistan'},
   {'role': 'assistant', 'content': 'Tashkent'}]}]

In [78]:
from datasets import Dataset
dataset = Dataset.from_list(messages_list)

In [79]:
def preprocessing(batch):
  templated_texts = []
  for conversation in batch['messages']:
    raw_text_sequence = tokenizer.apply_chat_template(
        conversation,
        tokenize = False
    )
    templated_texts.append(raw_text_sequence)
  print(templated_texts)
  return {'text':templated_texts}

In [80]:
tokenizer_dataset = dataset.map(preprocessing, batched = True)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

['<|im_start|>user\nCalculate 1+1<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n2<|im_end|>\n', '<|im_start|>user\nWhat is the capital of Uzbekistan<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nTashkent<|im_end|>\n']


In [81]:
print(tokenizer_dataset)

Dataset({
    features: ['messages', 'text'],
    num_rows: 2
})


In [82]:
training_args = TrainingArguments(
    output_dir='./lora_config',
    per_device_train_batch_size=2,      # Adjust based on VRAM (2-4 is safe for 24GB VRAM)
    gradient_accumulation_steps=4,     # Multiplies effective batch size (2 * 4 = 8)
    learning_rate=2e-4,

    logging_strategy="epoch",          # Tells the Trainer to print out stats every epoch

    num_train_epochs=10,                # Total training runs over the whole dataset
    optim="adamw_8bit",                # 8-bit optimizer cuts memory consumption drastically
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=True,                         # Set to True if your GPU doesn't support BF16
    save_strategy="epoch",             # Keeps saves synchronized with your logs
    report_to="none"
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [83]:
trainer = SFTTrainer(
    model = peft_model,
    train_dataset=tokenizer_dataset,
    args = training_args
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

In [84]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
1,5.689517
2,5.689517
3,3.928324
4,2.851406
5,2.254458
6,1.844748
7,1.620832
8,1.512613
9,1.430565
10,1.401554


TrainOutput(global_step=10, training_loss=2.822353649139404, metrics={'train_runtime': 22.3123, 'train_samples_per_second': 0.896, 'train_steps_per_second': 0.448, 'total_flos': 1344798720000.0, 'train_loss': 2.822353649139404, 'epoch': 10.0})

Using model for later inference

In [85]:
from peft import PeftModel
inference_model = PeftModel.from_pretrained(model, "./lora_config/checkpoint-6")

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [88]:
message =[{
    "role":'user',
    'content':"Calculate 3+3"
}]

tokenized = tokenizer.apply_chat_template(message, tokenize = False)
inputs = tokenizer(tokenized, return_tensors = 'pt').to(device)
outputs = model.generate(**inputs)

In [91]:
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
Calculate 3+3

</think>

6
